# Customer Intelligence & Support Copilot
## End-to-end notebook (Parts A-D)

**Scenario:** synthetic grocery quick-commerce data. The notebook shows how data is fetched, audited, cleaned, validated, stored, modelled and then consumed by the support-copilot components. It calls the same modules as the command-line scripts.

**Run it** from a clean kernel with *Restart & Run All*. Install `requirements.txt` first. All data is fictional; the LLM is the local `mock_llm.py` (no API key, no network).

> The notebook is saved **without outputs** so no stale numbers can mislead. Every figure appears only after you run the cells. Reference figures from the original capstone brief were not available when this notebook was written, so results are compared with the checkpoints in the README, not with the brief.

## 0. Repository map and flow

`generate_data.py` -> `data/raw/*.csv` -> audit -> clean -> `data/clean/*.csv` + `data/instamart.db` -> SQL/pandas analysis -> RFM segments + lapse model -> PII-safe ticket triage -> policy RAG + refund agent.

In [ ]:
from pathlib import Path
import os, sys, sqlite3, subprocess
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "generate_data.py").exists():
    ROOT = ROOT.parent                       # opened from notebooks/
assert (ROOT / "generate_data.py").exists(), "run from the repository root or notebooks/"
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
pd.set_option("display.width", 200, "display.max_columns", 30)
print("Project root:", ROOT)

## 1. Generate the raw data
`generate_data.py` is deterministic (seed 42) and is used exactly as supplied.

In [ ]:
subprocess.run([sys.executable, "generate_data.py"], cwd=ROOT, check=True)
print("Raw files:", sorted(p.name for p in (ROOT / "data" / "raw").iterdir()))

## 2. Part A - audit the raw exports
Blank strings are counted as missing. Delivery time is blank for cancelled orders (correct: never delivered) *and* for some completed ones (a data gap).

In [ ]:
from src import data_pipeline as dp
audit_report = dp.audit()

## 3. Part A - clean, store, validate
Rules, in order: parse `order_ts`; drop exact duplicates; keep the latest row per `order_id`; numeric `order_amount`; fill invalid amounts from price x quantity x discount; negative delivery -> missing; fill missing delivery **for completed orders only** with the product-category median; **cancelled orders keep a missing delivery time**.

In [ ]:
frames = dp.clean_data()
for name, df in frames.items():
    print(name, df.shape)
frames["orders"].head(3)

In [ ]:
assert dp.validate()          # prints PASS lines, then exactly: ALL CHECKPOINTS PASSED

## 4. Part A - SQL and pandas analysis
All seven queries in `sql/queries.sql` run with `pandas.read_sql_query` and are cross-checked against independent pandas calculations. Q2 is rebuilt with `pd.pivot_table(..., margins=True)`.

In [ ]:
from src import sql_analysis as sa
assert sa.main()

In [ ]:
print((ROOT / "outputs" / "eda_notes.md").read_text())

## 5. Part B - RFM, K-means, PCA
Snapshot `2025-10-01`. RFM is scaled with `StandardScaler`; K-means uses `n_init=10`, `random_state=42` for K=2..8. K is chosen from {3, 4, 5}. Cluster numbers are labels, not rankings, so names come from each cluster's profile.

In [ ]:
from src import customer_models as cm
rfm, diagnostics, profile, pca_var = cm.rfm_and_segments()
display(diagnostics.round(4))
display(profile.round(2))
print("PCA explained variance:", pca_var.round(4), "total:", round(float(pca_var.sum()), 4))

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 5)); plt.imshow(plt.imread(ROOT / "outputs" / "pca_segments.png")); plt.axis("off"); plt.show()

### Lapse prediction and leakage control
Cohort: customers with at least one completed order before **2025-09-01**. Label: no completed order in September. Features use only orders **before** the cutoff; imputation, scaling and encoding live inside a `Pipeline`, so they are fitted on training rows only. Segment labels are never features.

In [ ]:
summary = cm.build_lapse_model()
rows = {**summary["models"], **summary["baselines"]}
display(pd.DataFrame({k: {m: v[m] for m in ("precision", "recall", "f1", "accuracy")} for k, v in rows.items()}).T.round(3))
print("Selected:", summary["selected_model"], "-", summary["selection_rule"])
for k, v in rows.items():
    print(f"{k:>38} confusion matrix [[TN, FP], [FN, TP]] =", v["confusion_matrix"])

In [ ]:
os_ = summary["oversampling"]
print({k: os_[k] for k in ("train_rows_before", "train_rows_after", "train_lapsed_before", "train_lapsed_after")})
display(pd.DataFrame({m: {x: os_[m][x] for x in ("precision", "recall", "f1")} for m in ("logistic", "forest")}).T.round(3))
print("Reloaded model gives identical predictions:", summary["reload_predictions_identical"])
summary["retention_list"]

### Discussion: leakage
1. Which dates may enter the lapse features? Only those before 2025-09-01.
2. Why is the retention list *illustrative*? The Premium flag comes from RFM over the **whole** window (April-September), so it already "knows" September. It is a post-hoc label, not a leakage-free evaluation.
3. Why can accuracy mislead? About 38% of customers lapse; the "always lapsed" baseline already reaches F1 0.555, so judge a model against baselines, not in isolation.

## 6. Part C - ticket triage with guardrails
Order of defence: empty/over-long tickets are blocked **before** any model call; email -> phone -> order id are redacted; `assert_no_pii` checks every message (system and user) before every call, including retries; transient errors are retried with exponential backoff and jitter; outputs are validated against the four allowed categories; anything doubtful goes to a human.

In [ ]:
from src import triage as tr
few, few_s = tr.run_pipeline(tr.FEW, save_path=ROOT / "outputs" / "triage_results.csv")
zero, zero_s = tr.run_pipeline(tr.ZERO, save_path=ROOT / "outputs" / "triage_zero_shot.csv")
assert tr.validate_results(few) and tr.validate_results(zero)
display(pd.DataFrame({"few_shot": few_s, "zero_shot": zero_s}).loc[["accuracy_of_sent", "accuracy_of_classified", "model_calls", "retries", "tokens", "estimated_cost_usd"]])
print("few-shot status counts :", few_s["status_counts"])
print("few-shot routes        :", few_s["route_counts"])
print("redactions             :", few_s["redactions"])
few.head()

## 7. Part D - policy RAG
Policies are chunked into 40-word windows with a 10-word overlap, indexed with TF-IDF and ranked by cosine similarity. `answer()` sends exactly two messages (a `POLICY ASSISTANT` system message, then `CONTEXT` + `QUESTION`) to the mock model.

In [ ]:
from src import policy_rag as pr
rag = pr.PolicyRAG()
print("Chunks per document:", rag.chunk_counts(), "| total:", len(rag.chunks))
ev = pr.evaluate(rag)
display(ev[["question", "expected_doc", "retrieved_doc", "top1_doc", "fact_found", "fallback"]])
print(pr.summarize(ev))
print(pr.answer("Who must approve a refund above INR 1000?", rag)["answer"])

In [ ]:
cmp, failures = pr.compare_overlap()
print({k: (v["chunks"], v["expected_fact_found"]) for k, v in cmp.items()}, "<- (chunks, facts found out of 12)")
failures[["question", "expected_doc", "retrieved_doc", "fact_found", "answer"]]

## 8. Part D - refund agent
Five tools (`get_order`, `search_policy`, `calculate_refund`, `issue_refund`, `escalate_to_human`) sit in `TOOL_MAP`. `issue_refund` and `escalate_to_human` re-check the order in the database and are idempotent per order, so a refund cannot be paid twice. The loop is bounded (6 iterations, token limit, timeout) and checkpoints after every tool call. The cell below runs S1-S7, the token-limit and timeout demos, and the crash-and-resume demos.

In [ ]:
from src import run_scenarios as rs
assert rs.main()

In [ ]:
print((ROOT / "outputs" / "refund_ledger.jsonl").read_text())

## 9. Generated artifacts
Delete `data/clean/`, `data/instamart.db` and `outputs/`, then rerun the documented commands to prove the project rebuilds from source.

In [ ]:
for folder in ["data/clean", "outputs"]:
    print("\n" + folder, sorted(p.name for p in (ROOT / folder).iterdir()))
print("Database exists:", (ROOT / "data" / "instamart.db").exists())